In [2]:
import os
import sys
import json

In [3]:
dir_base = "../../"
print(os.path.abspath(dir_base))

sys.path.append(os.path.abspath(dir_base))

c:\Users\grabn\Documents\CGV_2024\xeegVis_alll\all-in-on-eeg


In [4]:
from backend.ml.data_utils.load_data import load_metadata
from backend.ml.data_utils.prepare_data import split_participants_min_per_class
from backend.ml.train import train_save_model
from backend.ml.model_vars import PRETRAINED_MODEL_DIR

In [5]:
def train():
    """
    Trains the xeegmodel.
    Saves the model.
    """
    dir_data = os.path.join("data", "datasets", "ds004504")

    # probably add timestamp to the model name to keep track of different runs
    modelname = "xeegnet_model_v1.pt"
    model_path = PRETRAINED_MODEL_DIR / modelname

    df_metadata = load_metadata(dir_data=dir_data)
    participants_ids_train, participants_ids_val, participants_ids_test = split_participants_min_per_class(
        df_metadata, ratios=(0.6, 0.2, 0.2), id_col="participant_id_int", random_state=42
    )

    model, df_metadata = train_save_model(
        model_path=model_path,
        dir_data=dir_data,
        df_metadata=df_metadata,
        participant_ids_train=participants_ids_train,
        participant_ids_val=participants_ids_val,
        # n_max=1000,
    )
    return model, df_metadata

In [6]:
dir_data = os.path.join(dir_base, "data", "datasets", "ds004504")

# probably add timestamp to the model name to keep track of different runs
df_metadata_orig = load_metadata(dir_data=dir_data)
participants_splits_path = os.path.join(dir_base, "backend\ml\data_splits.json")

# load json file participants splits
with open(participants_splits_path, "r") as f:
    participants_splits = json.load(f)

<>:5: SyntaxWarning: "\m" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\m"? A raw string is also an option.
<>:5: SyntaxWarning: "\m" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\m"? A raw string is also an option.
C:\Users\grabn\AppData\Local\Temp\ipykernel_10412\661543134.py:5: SyntaxWarning: "\m" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\m"? A raw string is also an option.
  participants_splits_path = os.path.join(dir_base, "backend\ml\data_splits.json")


In [ ]:
# i = 0

run_training = False

for i in range(0, 5):
    print(f"Processing split {i}...")

    participants_ids_train = participants_splits[str(i)]["train"]
    participants_ids_val = participants_splits[str(i)]["val"]
    participants_ids_test = participants_splits[str(i)]["test"]

    df_metadata = df_metadata_orig.copy(deep=True)
    df_metadata.loc[df_metadata["participant_id_int"].isin(participants_ids_train), "datasplit"] = "train"
    df_metadata.loc[df_metadata["participant_id_int"].isin(participants_ids_val), "datasplit"] = "val"
    df_metadata.loc[df_metadata["participant_id_int"].isin(participants_ids_test), "datasplit"] = "test"

    modelname = f"xeegnet_model_v20{i}.pt"
    model_path = PRETRAINED_MODEL_DIR / modelname

    if run_training:
        model, df_metadata = train_save_model(
            model_path=model_path,
            dir_data=dir_data,
            df_metadata=df_metadata,
            participant_ids_train=participants_ids_train,
            participant_ids_val=participants_ids_val,
            # n_max=1000,
        )

Processing split 0...
Processing split 1...
Processing split 2...
Processing split 3...
Processing split 4...


## Test models


In [ ]:
## load model and metadata
import torch
from backend.ml.model import build_xeegnet
from backend.ml.data_utils.load_data import *
from backend.ml.data_utils.prepare_data import *


<All keys matched successfully>

In [18]:
n_participants = df_metadata["participant_id_int"].max()
participants_ids = list(range(1, n_participants + 1))

df_eeg = load_multiple_eegfiles(
    dir_data=dir_data,
    participant_ids=participants_ids,  # , 43
    gen_path_func=gen_filename,
    df_metadata=df_metadata,
    n_max=1000,
)

Total memory usage of combined df_eeg: 73.85 MB


In [25]:
x_columns = [
    "time",
    "Fp1",
    "Fp2",
    "F3",
    "F4",
    "C3",
    "C4",
    "P3",
    "P4",
    "O1",
    "O2",
    "F7",
    "F8",
    "T3",
    "T4",
    "T5",
    "T6",
    "Fz",
    "Cz",
    "Pz",
]

downsample = True

nb_classes = 3
Chans = 19
srate = 125 if downsample else 250
workers = 0
batchsize = 64

window = 4
sample_length = int(srate * window)


In [50]:
model = build_xeegnet()

i = 0
modelname = f"xeegnet_model_v20{i}.pt"
model_path = PRETRAINED_MODEL_DIR / modelname

# load the model state dict
model.load_state_dict(torch.load(model_path))

# model.state_dict()
# model = torch.load(model_path)

<All keys matched successfully>

In [51]:
participants_ids_train = participants_splits[str(i)]["train"]
participants_ids_val = participants_splits[str(i)]["val"]
participants_ids_test = participants_splits[str(i)]["test"]

train_ids = [f"sub-{str(id).zfill(3)}" for id in participants_ids_train]
val_ids = [f"sub-{str(id).zfill(3)}" for id in participants_ids_val]
test_ids = [f"sub-{str(id).zfill(3)}" for id in participants_ids_test]

df_train = df_eeg[df_eeg["participant_id"].isin(train_ids)]
df_val = df_eeg[df_eeg["participant_id"].isin(val_ids)]
df_test = df_eeg[df_eeg["participant_id"].isin(test_ids)]

In [52]:
x_train, y_train = reshape_eeg_multipl(
    df_train, x_columns=x_columns, y_column="diagnosis", Chans=Chans, sample_length=sample_length
)
x_val, y_val = reshape_eeg_multipl(
    df_val, x_columns=x_columns, y_column="diagnosis", Chans=Chans, sample_length=sample_length
)
x_test, y_test = reshape_eeg_multipl(
    df_test, x_columns=x_columns, y_column="diagnosis", Chans=Chans, sample_length=sample_length
)

# trainloader = DataLoader(list(zip(x_train, y_train)), batch_size=batchsize, shuffle=True, num_workers=workers)
# valloader = DataLoader(list(zip(x_val, y_val)), batch_size=batchsize, shuffle=False, num_workers=workers)
# testloader = DataLoader(list(zip(x_test, y_test)), batch_size=batchsize, shuffle=False, num_workers=workers)

  0%|          | 0/55 [00:00<?, ?it/s]

100%|██████████| 18/18 [00:00<00:00, 74.91it/s]


In [53]:
test_output.shape

torch.Size([360, 3])

In [62]:
test_output = model(x_train.float())

test_output

# get percentages per class
test_pred = torch.argmax(test_output, dim=1)

print(test_pred)


tensor([1, 1, 0,  ..., 0, 2, 2])


In [63]:
print(y_train)

tensor([1, 1, 1,  ..., 2, 2, 2])


In [ ]:
print(sum(y_train == 0))
print(sum(y_train == 1))
print(sum(y_train == 2))
###
# print reltive distribution of classes in train, val, test
print(sum(y_train == 0) / len(y_train))
print(sum(y_train == 1) / len(y_train))
print(sum(y_train == 2) / len(y_train))


tensor(340)
tensor(480)
tensor(280)
tensor(0.3091)
tensor(0.4364)
tensor(0.2545)


In [68]:
print(sum(y_test == 0))
print(sum(y_test == 1))
print(sum(y_test == 2))

tensor(120)
tensor(120)
tensor(120)


In [ ]:
model = build_xeegnet()

i = 1
modelname = f"xeegnet_model_v20{i}.pt"
model_path = PRETRAINED_MODEL_DIR / modelname

# load the model state dict
model.load_state_dict(torch.load(model_path))
model.state_dict()

OrderedDict([('encoder.conv1.weight',
              tensor([[[[-4.5299e-04, -5.4646e-04, -6.4812e-04, -7.5805e-04, -8.7554e-04,
                         -9.9872e-04, -1.1244e-03, -1.2480e-03, -1.3636e-03, -1.4640e-03,
                         -1.5414e-03, -1.5877e-03, -1.5953e-03, -1.5578e-03, -1.4708e-03,
                         -1.3327e-03, -1.1454e-03, -9.1472e-04, -6.5114e-04, -3.6953e-04,
                         -8.9225e-05,  1.6652e-04,  3.7132e-04,  4.9677e-04,  5.1380e-04,
                          3.9436e-04,  1.1312e-04, -3.5063e-04, -1.0111e-03, -1.8745e-03,
                         -2.9371e-03, -4.1846e-03, -5.5911e-03, -7.1189e-03, -8.7188e-03,
                         -1.0331e-02, -1.1887e-02, -1.3310e-02, -1.4521e-02, -1.5437e-02,
                         -1.5979e-02, -1.6072e-02, -1.5649e-02, -1.4658e-02, -1.3059e-02,
                         -1.0832e-02, -7.9754e-03, -4.5107e-03, -4.8019e-04,  4.0522e-03,
                          9.0018e-03,  1.4266e-02,  1.9725e-02